<a href="https://colab.research.google.com/github/mayankjalann/24CSB0B27-COMPILER_DESIGN/blob/main/t5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Nayi notebook ki pehli cell mein ye daal aur run kar:
!pip uninstall -y transformers datasets accelerate sentencepiece
!pip install transformers==4.36.2 datasets==2.14.7 accelerate==0.25.0 sentencepiece

Found existing installation: transformers 5.4.0
Uninstalling transformers-5.4.0:
  Successfully uninstalled transformers-5.4.0
Found existing installation: datasets 4.8.4
Uninstalling datasets-4.8.4:
  Successfully uninstalled datasets-4.8.4
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Found existing installation: sentencepiece 0.2.1
Uninstalling sentencepiece-0.2.1:
  Successfully uninstalled sentencepiece-0.2.1
  Using cached transformers-4.36.2-py3-none-any.whl.metadata (126 kB)
  Using cached datasets-2.14.7-py3-none-any.whl.metadata (19 kB)
  Using cached accelerate-0.25.0-py3-none-any.whl.metadata (18 kB)
  Using cached sentencepiece-0.2.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (10 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached tokenizers-0.15.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
Using ca

In [ ]:
!pip uninstall -y peft

Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1


In [ ]:
!pip install --upgrade sympy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 112.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.5/527.5 kB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 616.3/616.3 kB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 113.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 117.6 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.2
    Uninstalling hf-xet-1.3.2:
      Successfully uninstalled hf-xet-1.3.2
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.

In [ ]:
import os
import re
import torch
import pyarrow.ipc as ipc
from datasets import Dataset
from google.colab import drive
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Trainer, TrainingArguments

# ==========================================
# 1. Drive Mount
# ==========================================
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
    print("✅ Drive Mounted.")

# ==========================================
# 2. Load Tokenizer & EPOCH 1 CHECKPOINT
# ==========================================
# 🔥 FIX 1: Removed 'tmp-' from the path. Make sure you renamed the folder in Drive!
checkpoint_path = "/content/drive/MyDrive/ai_summarization_project/codet5_final_blindfold/tmp-checkpoint-5625"

print("🔄 Loading Brain & Tokenizer from Epoch 1 Checkpoint...")
# 🔥 FIX 2: Used AutoTokenizer & AutoModel instead of Roberta/T5 base classes
# This ensures your custom AST tokens are loaded perfectly without type errors!
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint_path)
print("✅ Epoch 1 Model & Tokenizer Ready.")

# ==========================================
# 3. Dataset Load
# ==========================================
arrow_path = "/content/drive/MyDrive/ai_summarization_project/value_aware_ast_data/data-00000-of-00001.arrow"
with open(arrow_path, "rb") as f:
    df = ipc.RecordBatchStreamReader(f).read_all().to_pandas()
dataset = Dataset.from_pandas(df)
print(f"✅ Dataset Loaded: {len(dataset)} rows")

# ==========================================
# 4. THE MASTER FIX: HIDE RECURSION LEAKS
# ==========================================
def process_data(batch):
    cleaned_ast_sequences = []

    for ast_str in batch["ast_sequence"]:
        s = str(ast_str)

        # Step A: Find the function name
        match = re.search(r'FunctionDef\s+(\w+)', s)
        if match:
            func_name = match.group(1)
            # Step B: Mask it
            s = re.sub(rf'\b{func_name}\b', 'MASK_FUNC', s)

        cleaned_ast_sequences.append("summarize: " + s)

    inputs = cleaned_ast_sequences
    targets = [str(x).split('\n')[0].split('. ')[0].split(':param')[0].strip() for x in batch["docstring"]]

    model_inputs = tokenizer(inputs, max_length=512, padding="max_length", truncation=True)
    labels = tokenizer(text_target=targets, max_length=64, padding="max_length", truncation=True)

    labels_with_ignore_index = []
    for label_seq in labels["input_ids"]:
        labels_with_ignore_index.append([label if label != tokenizer.pad_token_id else -100 for label in label_seq])

    model_inputs["labels"] = labels_with_ignore_index
    return model_inputs

print("🔄 Erasing Recursive Function Leaks & Tokenizing...")
tokenized_dataset = dataset.map(process_data, batched=True, remove_columns=dataset.column_names)
split_data = tokenized_dataset.train_test_split(test_size=0.1)
print("✅ Tokenization Complete!")

# ==========================================
# 5. RESUME TRAINING
# ==========================================
args = TrainingArguments(
    output_dir="/content/drive/MyDrive/ai_summarization_project/codet5_final_blindfold",
    evaluation_strategy="epoch",    # <--- ISKO WAPAS PURANA NAAM DE DIYA
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    num_train_epochs=2,
    save_strategy="epoch",
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

# 🔥 FIX 4: Trainer ab automatic hardware bind karega (Accelerate error bypass)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=split_data["train"],
    eval_dataset=split_data["test"],
)

print("⚡ Resuming Training! Now recursion is strictly hidden.")
trainer.train()

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


🔄 Loading Brain & Tokenizer from Epoch 1 Checkpoint...


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


OSError: Incorrect path_or_model_id: '/content/drive/MyDrive/ai_summarization_project/codet5_final_blindfold/tmp-checkpoint-5625'. Please provide either the path to a local folder or the repo_id of a model on the Hub.

In [ ]:
import torch
print("Torch is loading from:", torch.__file__)

Torch is loading from: /usr/local/lib/python3.12/dist-packages/torch/__init__.py


📊 TRACING YOUR LOSS HISTORY 📊
--------------------------------------------------
Step       | Epoch      | Training Loss   | Validation Loss
--------------------------------------------------
500        | 0.09       | 2.0536          | -
1000       | 0.18       | 2.1006          | -
1500       | 0.27       | 2.0911          | -
2000       | 0.36       | 2.112           | -
2500       | 0.44       | 2.0765          | -
3000       | 0.53       | 2.1205          | -
3500       | 0.62       | 2.0821          | -
4000       | 0.71       | 2.1088          | -
4500       | 0.8        | 2.1183          | -
5000       | 0.89       | 2.1149          | -
5500       | 0.98       | 2.0872          | -
5625       | 1.0        | -               | 1.8193001747131348


In [ ]:
import ast
import re
import torch, os
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
    print("✅ Drive Mounted.")

# ==========================================
# 1. THE AST CONVERTER
# ==========================================
def python_to_ast_string(code_str):
    try:
        tree = ast.parse(code_str)
        tokens = []

        def traverse(node):
            if isinstance(node, ast.AST):
                tokens.append(node.__class__.__name__)
                if hasattr(node, 'name') and isinstance(node.name, str):
                    tokens.append(node.name)
                elif hasattr(node, 'id') and isinstance(node.id, str):
                    tokens.append(node.id)
                elif hasattr(node, 'arg') and isinstance(node.arg, str):
                    tokens.append(node.arg)

                for child in ast.iter_child_nodes(node):
                    traverse(child)

        traverse(tree)
        return " ".join(tokens)
    except Exception as e:
        return f"Error parsing code: {e}"

# ==========================================
# 2. LOAD THE NEW "TRUE LOGIC" BRAIN
# ==========================================
print("🔄 Booting up the Smart Compiler AI (Strict Recursion Blindfold)...")

# 🔥 THE FIX: Tokenizer purane folder se jahan AST tokens safe hain (Epoch 2)
tokenizer_path = "/content/drive/MyDrive/ai_summarization_project/codet5_final_blindfold/checkpoint-5625"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

# 🔥 THE FIX: Model nayi memory wala jisme 4 Epochs ki training hai
model_path = "/content/drive/MyDrive/ai_summarization_project/codet5_final_blindfold/checkpoint-11250"
model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to("cuda" if torch.cuda.is_available() else "cpu")

model.eval()
print("✅ Brain Loaded & Ready!\n")

# ==========================================
# 3. THE MAGIC PREDICTION FUNCTION
# ==========================================
def generate_smart_summary(raw_code):
    print("-" * 50)
    print("💻 RAW CODE INPUT:")
    print(raw_code.strip())
    print("-" * 50)

    raw_ast = python_to_ast_string(raw_code)

    match = re.search(r'FunctionDef\s+(\w+)', raw_ast)
    if match:
        func_name = match.group(1)
        masked_ast = re.sub(rf'\b{func_name}\b', 'MASK_FUNC', raw_ast)
    else:
        masked_ast = raw_ast

    print(f"🛡️ CLEANED AST (Fed to Model): {masked_ast[:100]}...")

    input_text = "summarize: " + masked_ast
    inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).to(model.device)

    # Relaxed generation parameters taaki sentence poora ho
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_length=64,
            min_length=5,
            num_beams=5,
            repetition_penalty=2.0,
            early_stopping=True
        )

    summary = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if summary:
        summary = summary[0].upper() + summary[1:]
        if not summary.endswith('.'):
            summary += '.'

    print(f"\n✨ AI SUMMARY OUTPUT: \n👉 {summary}\n")
    return summary


# ==========================================
# 4. ORIGINAL CORE TESTS (1-3)
# ==========================================
demo_codes = [
"""
def do_magic(n):
    if n == 0 or n == 1:
        return 1
    return n * do_magic(n - 1)
""",
"""
def xyz_algo(a, b):
    while b:
        a, b = b, a % b
    return a
""",
"""
def search_thing(arr, target):
    l, r = 0, len(arr) - 1
    while l <= r:
        m = (l + r) // 2
        if arr[m] == target: return m
        elif arr[m] < target: l = m + 1
        else: r = m - 1
    return -1
"""
]

# ==========================================
# 5. ORIGINAL TRICK TESTS (4-8)
# ==========================================
more_demo_codes = [
"""
def check_it(s):
    s = s.lower()
    return s == s[::-1]
""",
"""
def compute_val(base, p):
    res = 1
    for _ in range(p):
        res *= base
    return res
""",
"""
def get_top_item(arr):
    if not arr: return None
    best = arr[0]
    for val in arr:
        if val > best:
            best = val
    return best
""",
"""
def is_special_time(y):
    if y % 400 == 0: return True
    if y % 100 == 0: return False
    return y % 4 == 0
""",
"""
def process_chars(t):
    cnt = 0
    for c in t:
        if c.lower() in 'aeiou':
            cnt += 1
    return cnt
"""
]

# ==========================================
# 6. HEAVYWEIGHT TESTS (9-12)
# ==========================================
complex_demo_codes = [
"""
def explore_paths(g, start, seen=None):
    if seen is None:
        seen = set()
    if start not in seen:
        seen.add(start)
        for nxt in g[start]:
            explore_paths(g, nxt, seen)
    return seen
""",
"""
def xyz(adj, root):
    visited = [root]
    q = [root]
    while q:
        curr = q.pop(0)
        for nxt in adj[curr]:
            if nxt not in visited:
                visited.append(nxt)
                q.append(nxt)
    return visited
""",
"""
def xyz(data):
    sz = len(data)
    for i in range(sz):
        for j in range(0, sz - i - 1):
            if data[j] > data[j + 1]:
                data[j], data[j + 1] = data[j + 1], data[j]
    return data
""",
"""
def xyz(m1, m2):
    r1, c1 = len(m1), len(m1[0])
    r2, c2 = len(m2), len(m2[0])
    out = [[0 for _ in range(c2)] for _ in range(r1)]
    for i in range(r1):
        for j in range(c2):
            for k in range(c1):
                out[i][j] += m1[i][k] * m2[k][j]
    return out
"""
]

# ==========================================
# 🔥 7. NEW ULTIMATE CHALLENGE TESTS (13-18)
# ==========================================
extra_challenge_codes = [
"""
def xyz(n):
    if n <= 1:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True
""",
"""
def xyz(n, memo={}):
    if n in memo: return memo[n]
    if n <= 1: return n
    memo[n] = sequence_maker(n-1, memo) + sequence_maker(n-2, memo)
    return memo[n]
""",
"""
def xyz(s1, s2):
    if len(s1) != len(s2): return False
    counts = {}
    for c in s1:
        counts[c] = counts.get(c, 0) + 1
    for c in s2:
        if c not in counts or counts[c] == 0:
            return False
        counts[c] -= 1
    return True
""",
"""
def xyz(node, res=None):
    if res is None:
        res = []
    if node:
        walk_nodes(node.left, res)
        res.append(node.value)
        walk_nodes(node.right, res)
    return res
""",
"""
def fxyz(arr):
    left, right = 0, len(arr) - 1
    while left < right:
        arr[left], arr[right] = arr[right], arr[left]
        left += 1
        right -= 1
    return arr
""",
"""
def xyz(arr):
    for i in range(1, len(arr)):
        key = arr[i]
        j = i - 1
        while j >= 0 and key < arr[j]:
            arr[j + 1] = arr[j]
            j -= 1
        arr[j + 1] = key
    return arr
"""
]

# --- RUNNING ALL SUITES ---
suites = [
    ("CORE BLINDFOLD TESTS", demo_codes, 1),
    ("MORE BLINDFOLD TESTS", more_demo_codes, 4),
    ("HEAVYWEIGHT BLINDFOLD TESTS", complex_demo_codes, 9),
    ("ULTIMATE CHALLENGE TESTS (NEW)", extra_challenge_codes, 13)
]

for suite_name, test_group, start_idx in suites:
    print(f"\n🚀 RUNNING {len(test_group)} {suite_name}...\n")
    for i, code in enumerate(test_group, start_idx):
        print(f"\n{'='*50}\n🧪 TEST {i}:")
        generate_smart_summary(code)

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


KeyboardInterrupt: 

In [1]:
!pip install evaluate rouge_score sacrebleu

In [2]:
!pip uninstall -y transformers datasets accelerate sentencepiece evaluate rouge_score sacrebleu
!pip install transformers==4.36.2 datasets==2.14.7 accelerate==0.25.0 sentencepiece evaluate rouge_score sacrebleu

Found existing installation: transformers 4.39.3
Uninstalling transformers-4.39.3:
  Successfully uninstalled transformers-4.39.3
Found existing installation: datasets 4.8.4
Uninstalling datasets-4.8.4:
  Successfully uninstalled datasets-4.8.4
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Found existing installation: sentencepiece 0.2.1
Uninstalling sentencepiece-0.2.1:
  Successfully uninstalled sentencepiece-0.2.1
Found existing installation: evaluate 0.4.6
Uninstalling evaluate-0.4.6:
  Successfully uninstalled evaluate-0.4.6
Found existing installation: rouge_score 0.1.2
Uninstalling rouge_score-0.1.2:
  Successfully uninstalled rouge_score-0.1.2
Found existing installation: sacrebleu 2.6.0
Uninstalling sacrebleu-2.6.0:
  Successfully uninstalled sacrebleu-2.6.0
  Using cached transformers-4.36.2-py3-none-any.whl.metadata (126 kB)
  Using cached datasets-2.14.7-py3-none-any.whl.metadata (19 kB)
  Using c

In [4]:
!pip uninstall -y transformers datasets accelerate tokenizers sentencepiece
!pip install transformers==4.36.2 datasets==2.14.7 accelerate==0.25.0 tokenizers==0.15.0 sentencepiece

Found existing installation: transformers 4.36.2
Uninstalling transformers-4.36.2:
  Successfully uninstalled transformers-4.36.2
Found existing installation: datasets 2.14.7
Uninstalling datasets-2.14.7:
  Successfully uninstalled datasets-2.14.7
Found existing installation: accelerate 0.25.0
Uninstalling accelerate-0.25.0:
  Successfully uninstalled accelerate-0.25.0
Found existing installation: tokenizers 0.15.2
Uninstalling tokenizers-0.15.2:
  Successfully uninstalled tokenizers-0.15.2
Found existing installation: sentencepiece 0.2.1
Uninstalling sentencepiece-0.2.1:
  Successfully uninstalled sentencepiece-0.2.1
  Using cached transformers-4.36.2-py3-none-any.whl.metadata (126 kB)
  Using cached datasets-2.14.7-py3-none-any.whl.metadata (19 kB)
  Using cached accelerate-0.25.0-py3-none-any.whl.metadata (18 kB)
  Using cached sentencepiece-0.2.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (10 kB)
Using cached transformers-4.36.2-py3-none-any.whl (8.2 MB)
U

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
import torch
from transformers import RobertaTokenizer, T5ForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"

model_path = "/content/drive/MyDrive/ai_summarization_project/codet5_final_blindfold/checkpoint-11250"
print("Exists?", os.path.isdir(model_path))
print("Files:", os.listdir(model_path)[:10] if os.path.isdir(model_path) else "Folder not found")

assert os.path.isdir(model_path), f"Model folder not found: {model_path}"

tokenizer = RobertaTokenizer.from_pretrained("Salesforce/codet5-base", use_fast=False)
model = T5ForConditionalGeneration.from_pretrained(model_path, local_files_only=True).to(device)
model.eval()

Mounted at /content/drive
Exists? True
Files: ['config.json', 'generation_config.json', 'model.safetensors', 'training_args.bin', 'optimizer.pt', 'scheduler.pt', 'rng_state.pth', 'trainer_state.json']


T5ForConditionalGeneration(
  (shared): Embedding(32100, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32100, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=768, out_features=3072, bias=False)
              (wo): Linear(in_features=3072, out_features=768, bias=False)
              (dropout): Dro

In [ ]:
import re
import torch
import numpy as np
import pyarrow.ipc as ipc
from datasets import Dataset
import evaluate
from tqdm import tqdm

print("🚀 Starting Evaluation Phase...")

# Metrics
bleu_metric = evaluate.load("sacrebleu")
rouge_metric = evaluate.load("rouge")

# Load dataset
arrow_path = "/content/drive/MyDrive/ai_summarization_project/value_aware_ast_data/data-00000-of-00001.arrow"

with open(arrow_path, "rb") as f:
    df = ipc.RecordBatchStreamReader(f).read_all().to_pandas()

dataset = Dataset.from_pandas(df)

# Preprocess
def preprocess(example):
    s = str(example["ast_sequence"])

    match = re.search(r'FunctionDef\s+(\w+)', s)
    if match:
        func_name = match.group(1)
        s = re.sub(rf'\b{func_name}\b', 'MASK_FUNC', s)

    input_text = "summarize: " + s

    target = str(example["docstring"]) \
        .split('\n')[0] \
        .split('. ')[0] \
        .split(':param')[0] \
        .strip()

    return input_text, target

print("🔄 Preparing data...")
inputs, targets = [], []

for ex in dataset:
    inp, tgt = preprocess(ex)
    inputs.append(inp)
    targets.append(tgt)

# 🔥 Optional speed test
# inputs = inputs[:200]
# targets = targets[:200]

# Generate predictions
print("⚡ Generating predictions...")
predictions = []
batch_size = 8
device = "cuda" if torch.cuda.is_available() else "cpu"

for i in tqdm(range(0, len(inputs), batch_size)):
    batch_inputs = inputs[i:i+batch_size]

    enc = tokenizer(
        batch_inputs,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **enc,
            max_length=64,
            num_beams=4
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    predictions.extend([x.strip() for x in decoded])

# Clean targets
targets_clean = [t.strip() for t in targets]
targets_bleu = [[t] for t in targets_clean]

# Metrics
print("📊 Computing metrics...")

bleu = bleu_metric.compute(predictions=predictions, references=targets_bleu)
rouge = rouge_metric.compute(predictions=predictions, references=targets_clean)

print("\n✅ FINAL RESULTS:")
print(f"BLEU: {bleu['score']:.4f}")
print(f"ROUGE-1: {rouge['rouge1']:.4f}")
print(f"ROUGE-2: {rouge['rouge2']:.4f}")
print(f"ROUGE-L: {rouge['rougeL']:.4f}")

🚀 Starting Evaluation Phase...
🔄 Preparing data...
⚡ Generating predictions...


 27%|██▋       | 1717/6250 [24:54<1:12:45,  1.04it/s]

🔄 Loading Tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

🔄 Loading Dataset...
🔄 Tokenizing and applying Blindfold...


Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

✅ Tokenizer and Dataset Ready! You are cleared to run the Trainer block.


NameError: name 'Seq2SeqTrainer' is not defined

In [ ]:
#interface

In [ ]:
!pip install flask flask-cors -q
!npm install -g localtunnel -q


⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
added 22 packages in 4s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸

🔄 Loading CodeT5 model and tokenizer from checkpoint...


Loading weights:   0%|          | 0/257 [00:02<?, ?it/s]

✅ Model ready on cpu!


🧪 Test summary: 


In [ ]:
import threading
import subprocess
import time
from flask import Flask, request, jsonify
from flask_cors import CORS

app = Flask(__name__)
CORS(app)

@app.route("/summarize", methods=["POST"])
def summarize():
    try:
        data = request.get_json(force=True)
        source_code = data.get("code", "").strip()
        if not source_code:
            return jsonify({"error": "No code provided."}), 400
        summary = generate_smart_summary(source_code)
        return jsonify({"summary": summary, "model": "CodeT5-Finetuned", "status": "ok"})
    except ValueError as e:
        return jsonify({"error": str(e)}), 422
    except Exception as e:
        return jsonify({"error": f"Internal error: {str(e)}"}), 500

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "device": device})

def run_flask():
    app.run(port=5000, use_reloader=False, debug=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()
time.sleep(2)

tunnel = subprocess.Popen(["lt", "--port", "5000"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
raw_output = tunnel.stdout.readline().decode("utf-8").strip()

if "your url is:" in raw_output:
    public_url = raw_output.split("your url is: ")[-1].strip()
else:
    public_url = raw_output

print("=" * 60)
print(f"🚀 API is LIVE at: {public_url}/summarize")
print("=" * 60)


 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


🚀 API is LIVE at: https://better-camels-switch.loca.lt/summarize


In [ ]:
# DEBUG CELL — paste this in Colab and run it
import requests

test_code = """
def xyz(arr):
    for i in range(1, len(arr)):
        key = arr[i]
        j = i - 1
        while j >= 0 and key < arr[j]:
            arr[j + 1] = arr[j]
            j -= 1
        arr[j + 1] = key
    return arr
"""

# Test 1: Direct function call (what your console shows)
print("=== DIRECT CALL ===")
direct = generate_smart_summary(test_code)

# Test 2: Via Flask API (what the UI calls)
print("=== VIA FLASK API ===")
resp = requests.post("http://127.0.0.1:5000/summarize", json={"code": test_code})
api_result = resp.json()["summary"]
print("API returned:", api_result)

print("\n=== MATCH? ===", direct == api_result)


=== DIRECT CALL ===
--------------------------------------------------
💻 RAW CODE INPUT:
def xyz(arr):
    for i in range(1, len(arr)):
        key = arr[i]
        j = i - 1
        while j >= 0 and key < arr[j]:
            arr[j + 1] = arr[j]
            j -= 1
        arr[j + 1] = key
    return arr
--------------------------------------------------
🛡️ CLEANED AST (Fed to Model): Module FunctionDef MASK_FUNC arguments arg arr For Name i Store Call Name range Load Constant Call N...

✨ AI SUMMARY OUTPUT: 
👉 Sort an array by key.

=== VIA FLASK API ===
--------------------------------------------------
💻 RAW CODE INPUT:
def xyz(arr):
    for i in range(1, len(arr)):
        key = arr[i]
        j = i - 1
        while j >= 0 and key < arr[j]:
            arr[j + 1] = arr[j]
            j -= 1
        arr[j + 1] = key
    return arr
--------------------------------------------------
🛡️ CLEANED AST (Fed to Model): Module FunctionDef MASK_FUNC arguments arg arr For Name i Store Call Nam

INFO:werkzeug:127.0.0.1 - - [15/Mar/2026 06:05:38] "POST /summarize HTTP/1.1" 200 -



✨ AI SUMMARY OUTPUT: 
👉 Sort an array by key.

API returned: Sort an array by key.

=== MATCH? === True


In [ ]:
import urllib.request
print(urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())


35.221.128.161


In [ ]:
import subprocess
subprocess.run("kill -9 $(lsof -ti:5000) 2>/dev/null || true", shell=True)
print("✅ Old server killed!")


Path exists: True
Files inside folder:
['data-00000-of-00001.arrow', 'state.json', 'dataset_info.json']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install datasets==2.18.0

  Using cached datasets-2.18.0-py3-none-any.whl.metadata (20 kB)
Using cached datasets-2.18.0-py3-none-any.whl (510 kB)
  Attempting uninstall: datasets
    Found existing installation: datasets 2.19.1
    Uninstalling datasets-2.19.1:
      Successfully uninstalled datasets-2.19.1


✅ Dataset Loaded Successfully
Columns: ['id', 'repo', 'path', 'func_name', 'original_string', 'language', 'code', 'code_tokens', 'docstring', 'docstring_tokens', 'sha', 'url', 'ast_sequence']

Sample AST:
Module FunctionDef settext arguments arg self arg text arg cls Constant Expr Constant Expr Call Attribute Name self Load Load Name TextContent Load keyword value Name text Load keyword cls Name cls Lo

Sample Docstring:
Set the text for this element.

        Arguments:
            text (str): The text
            cls (str): The class of the text, defaults to ``current`` (leave this unless you know what you are doing). There may be only one text content element of each class associated with the element.


Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

✅ Dataset ready for training


In [ ]:
!pip install -U accelerate==0.27.2

  Attempting uninstall: accelerate
    Found existing installation: accelerate 0.30.1
    Uninstalling accelerate-0.30.1:
      Successfully uninstalled accelerate-0.30.1


Using python: /usr/bin/python3

$ /usr/bin/python3 -m pip uninstall -y transformers tokenizers datasets accelerate huggingface_hub sentencepiece

$ /usr/bin/python3 -m pip install --no-cache-dir transformers==4.41.2 tokenizers==0.19.1 datasets==2.19.1 accelerate==0.30.1 huggingface_hub==0.36.2 sentencepiece

=== Versions installed ===

$ /usr/bin/python3 -m pip show transformers

$ /usr/bin/python3 -m pip show accelerate

$ /usr/bin/python3 -m pip show tokenizers

$ /usr/bin/python3 -m pip show datasets

$ /usr/bin/python3 -m pip show huggingface_hub

Installed. IMPORTANT: Now restart the runtime (Runtime -> Restart runtime) so the notebook uses the new packages.

After restart, run this quick verification cell to confirm the error is fixed:

import accelerate
print('accelerate:', accelerate.__version__)
from transformers import Trainer
print('Trainer import OK')

If that prints the accelerate version and Trainer import succeeds, your environment is fixed.



Accelerate version: 0.30.1


RuntimeError: Failed to import transformers.trainer because of the following error (look up to see its traceback):
cannot import name 'clear_device_cache' from 'accelerate.utils.memory' (/usr/local/lib/python3.12/dist-packages/accelerate/utils/memory.py)

RuntimeError: Failed to import transformers.trainer because of the following error (look up to see its traceback):
cannot import name 'clear_device_cache' from 'accelerate.utils.memory' (/usr/local/lib/python3.12/dist-packages/accelerate/utils/memory.py)

In [ ]:
trainer.train()

Found existing installation: transformers 4.38.2
Uninstalling transformers-4.38.2:
  Successfully uninstalled transformers-4.38.2
Found existing installation: tokenizers 0.15.2
Uninstalling tokenizers-0.15.2:
  Successfully uninstalled tokenizers-0.15.2
Found existing installation: datasets 4.6.1
Uninstalling datasets-4.6.1:
  Successfully uninstalled datasets-4.6.1
Found existing installation: huggingface_hub 0.36.2
Uninstalling huggingface_hub-0.36.2:
  Successfully uninstalled huggingface_hub-0.36.2


In [ ]:
!pip uninstall -y transformers tokenizers
!pip install transformers==4.38.2 tokenizers==0.15.2 sentencepiece